# 🕮 Blue Books Crawler — Runner Notebook

This notebook **runs the WHO Blue Books crawler** step by step and builds the
searchable database — no command-line experience needed. You edit a few clearly
marked settings, then press ▶ Run on each cell.

It walks you through the whole pipeline:

1. **Set up** Python + the browser engine the crawler uses.
2. **Configure** your settings (`config.json`).
3. **Log in** once in a real browser window and save the session.
4. **Discover** which Blue Books volumes your subscription exposes.
5. **Crawl** the volume(s) you choose (resumable, polite rate limits).
6. **Build the search index** (FTS5).
7. **Check** what you collected.
8. *(Optional)* **Copy the data into SQL Server.**

> ### ⚠️ Run this on YOUR computer, not in a cloud kernel
> Steps 3–5 open a **real Chromium browser** and use **your logged-in
> BlueBooksOnline session**. They must run **locally** (VS Code on your own
> machine), where a browser window can appear and reach the licensed site.
> A remote/headless kernel cannot log in for you.
>
> ### 📄 Licensing
> WHO/IARC tumour-classification content is copyrighted and subscription-gated.
> Use extracts for **private, licensed, non-commercial study** only; keep the
> resulting database private. The crawler uses polite delays by default.

## Local data parity (no online access)

The shared `bluebooks_local` pipeline builds all local reference/entity/evidence
tables and whole-page search documents from saved corpus data and local inputs.
It recovers complete saved-HTML section bodies while retaining existing section
IDs and image/reference associations; exceptions are recorded in build provenance.
Code-only morphology lines use the page title as a fallback term. Explicit OMIM
identifiers are stored separately in entity_identifiers, never as ICD-O evidence.
Entity tables include fifth- and sixth-edition books; original code formats are
preserved and edition is joined through books. The Step 7 spreadsheet examples
may focus on sixth-edition books, but the shared refresh covers both editions.
It can mirror those tables to SQL Server and verify every row value.
See [DATA_DICTIONARY.md](DATA_DICTIONARY.md) in the crawler folder for all tables,
their sources and refresh order. Agent activity is untouched; book 70 and crawl
bookkeeping differences are accepted. Back up before refresh/mirror.
Schema provisioning includes the recommended page/book/TOC lookup indexes,
preserves all existing relationships, and consolidates only the two reviewed
SQL Server redundant indexes after dependency checks. No new FT index is needed.
Do not run login/discovery/crawl or package-download cells for local validation.


## Term hygiene and indexed US/comparison values

Schema version 10 cleans term-list footnotes (dagger, double dagger, asterisk,
section/paragraph marks), leading bullets, trailing whitespace and list punctuation.
Scientific signs, numeric superscripts/subscripts, codes, raw HTML, section prose
and snippets remain intact. The saved-page parser and database INSERT/UPDATE
triggers prevent these artifacts from returning.

| Table | Source for TermUS and TermNormUS |
|---|---|
| entity_codes | term |
| entity_related_terms | term |
| page_icdo_codes | term_in_coding_section |
| page_term_mentions | term_original |
| page_entity_evidence | term_clean (existing evidence grouping key) |

`TermUS = UpperInitial(CleanBritishSpelling2(CleanDisplayTerm(source)))` and
`TermNormUS = UpperInitial(Clean2(TermUS))`. Both are stored Unicode values with separate
nonunique lookup indexes in **both databases**. Stored values plus automatic
maintenance are used rather than computed columns, so ordinary SQLite connections
can read them without evaluating a Python UDF. Writes to these tables must use
`local.connect_sqlite`, which registers the deterministic functions.

Clean2 uses the existing deployed SQL Server v3 definition, explicitly selected
by the maintainer on 2026-10-10, not the earlier repository version. Its exact
SQL replacement program also drives Python, preventing rule drift. Case is
preserved inside Clean2; display handling uppercases only an initial ASCII a-z.
Digits, Greek symbols, punctuation-led values and the remaining characters stay
unchanged. No synonym equivalence or code crosswalk is inferred.
TermNormUS is separate from legacy term_clean/term_original_norm/matched_term_norm;
those keys keep the legacy Clean rules but no longer retain footnote decoration.

For an existing snapshot, `local.repair_term_tables(SQLITE_PATH)` makes fresh
SQLite and SQL Server COPY_ONLY backups, repairs the five tables in place,
preserves genuine IDs/codes/provenance, removes only saved-HTML-confirmed
misplaced prose approved by the maintainer, and verifies all 23 managed row hashes.
Full Runner/Explorer refreshes and mirrors also maintain the new columns.
Notebook regeneration backs up every notebook before writing it.

Initial suffix audit (per database): 674 page_icdo_codes rows, 673
page_term_mentions rows and 558 page_entity_evidence rows. The named entity
tables had no dagger/asterisk suffixes, but do contain trailing punctuation.
Confirmed misplaced explanatory prose is removed from uncoded entity/related
term rows. Scientific parenthetical qualifiers remain. Contextual code terms
containing punctuation/prose are recovered from an unambiguous structured
assignment in coding/subtype sections, otherwise set to NULL; their code evidence
is preserved. This was the initial version-7 repair; version 9 recovers explicit
name/code pairs from the same saved section, including shared-code alternatives
and nested headings, without assigning the parent diagnosis to a contextual code.

Completed in both databases on 2026-10-11: 44 misplaced uncoded entity rows
and 41 related-term prose/heading rows removed; 465 contextual code terms
repaired (407 recovered names, 58 NULL rather than invented names).
All code assignments, raw saved evidence and genuine scientific qualifiers
were retained. Source values changed in 32 entity-code, 14 retained related-term,
1,279 contextual-code, 814 mention and 672 evidence rows.
All five tables have separate TermUS/TermNormUS indexes and automatic maintenance.
Validation found no empty derived keys or rule mismatches; 23-table row-value
parity passed. Initial notebook and database backups are in local_backups;
SQL Server COPY_ONLY backups are in the instance's default Backup directory.

Version-9 follow-up: all 58 NULL coding names were recoverable from explicit
saved-source pairs. Code IDs and assignments are retained; any absent shared-code
alternatives are added, not discarded. Display capitalization also covers
identifier terms, reference terms/keys, recovered names and proposed official names.
Original prose and workbook code-term evidence JSON remain verbatim.
Absent optional workbook preferred terms remain NULL; none are invented.
Use `local.repair_term_display(SQLITE_PATH)` for the backed-up display repair;
Clean2 itself is unchanged. Both engines and all 25 managed tables are verified.


In [ ]:
# Read-only checks: use the existing local database, no crawl or login.
from contextlib import closing
import bluebooks_local as local
with closing(local.connect_sqlite(local.ROOT / "bluebooks.db", readonly=True)) as term_db:
    print("Term hygiene/key mismatches:", local.term_hygiene.validate_terms(term_db))
    print("Sample TermUS / TermNormUS:", term_db.execute(
        "SELECT term, TermUS, TermNormUS FROM entity_codes "
        "WHERE term LIKE '%tumour%' LIMIT 5"
    ).fetchall())


## Recovered alternate terms and suggested ICD-O matches

Schema version 8 preserves valid names embedded in the removed prose.
New names are in `entity_related_terms` with `kind='other'`; provenance is
explicitly `AI_recovered_prose` in `entity_term_provenance`. The suggested
qualifier is synonym, related, subtype, grouper or other, separate from
terminology acceptability. Historical names and uncoded diagnoses remain searchable.

`entity_term_code_matches` holds edition-specific exact/synonym/broader/conditional
**proposed** matches, official reference terms, workbook row/hash evidence and
conditions. These never overwrite source codes or count as positive source-code
evidence. Review status defaults to proposed. Code availability does not prove a
patient diagnosis or registry reportability.

Use `v_entity_terms` for every term once, `v_entity_term_provenance` for every
source link, and `v_entity_term_code_matches` for comprehensive terms plus candidate
codes, including unmapped terms. `source_code` and `suggested_code` are separate.
Multiple candidate rows are intentional. Avoid an edition WHERE filter that
accidentally removes NULL/unmapped rows.

Related-terminology extraction splits semicolons only outside balanced
parentheses, preserving terms such as `t(15;19) carcinoma`. Qualified
Acceptable/Not recommended labels are separated from their term lists, while
cross-references and explanatory prose are not stored as terms. Full refresh
re-extracts these rows from saved page HTML.

Schema version 10 adds `v_page_code_terms`: editable `books.alias` book/version,
chapter plus `ch_order`, page title/URL, relevant sections, scalar TermUS/TermNormUS/candidateTermUS and ICDO32/ICDO4,
source/candidate kinds, subtype/qualifier/match/review metadata and conditions.
Each page/edition/code/TermUS pair occurs once; different terms sharing a code
remain separate. Only sections/metadata and page-level candidate topography
codes are concatenated. Site hints require confidence_score >= 0.9 and are
not verified term-specific or patient sites. Unmapped terms keep NULL codes.
`v_entity_term_code_matches.official_term` and `candidateTermUS` now display US
spelling; `official_term_source` preserves original reference wording.

### Deprecated legacy field: `is_subtype`

**Deprecated:** do not use `is_subtype` in new queries or extraction logic.
It remains exposed for compatibility only; prefer `term_role` and explicit
source provenance/relationship metadata. This section describes historical
behavior. It was a source-section flag, not an inferred biological
classification. In `entity_codes`, `1` meant the term came from a Blue Books
**Subtype(s)** section, including nested subtype sections; `0` meant it came
from the ordinary **ICD-O coding** section. `_parse_subtype_block` in the
companion `bb_entity_extract.py` set the legacy flag; `bluebooks_local.py`
stored it as 0/1.

For backward compatibility, `v_page_code_terms` still exposes this deprecated
field. Entity rows carry the stored flag; contextual `page_icdo_codes` rows
inherit it from a matching **page + raw code + TermUS** in `entity_codes`,
otherwise `0`. Historically, repeated occurrences used **`MAX(is_subtype)`**.
New consumers should use `term_role` and explicit provenance instead.

Recovered alternate terms in `entity_related_terms` appear with deprecated
`is_subtype=0`, even when their recovery metadata has `suggested_qualifier='subtype'`.
These deliberately express different things:

| Field | Meaning |
|---|---|
| Deprecated `is_subtype=1` | Legacy flag for extraction from a source subtype section |
| `suggested_qualifier='subtype'` | Proposed relationship for a recovered term |

**Deprecated `is_subtype=0` does not mean a diagnosis is not biologically a subtype.**
It means the legacy source-subtype flag was not established by this extraction pathway.

### `v_page_code_terms.terminology_kind` values

`terminology_kind` records where in the source page a term came from (provenance
only; not code validity, preferred-term, synonym or registry status):

| Value | Definition | Origin |
|---|---|---|
| `coding` | Term from the ICD-O coding or Subtype(s) section | `entity_codes` |
| `contextual` | Term-code pair found in section text | `page_icdo_codes` |
| `acceptable` | Related terminology labeled "Acceptable" | `entity_related_terms.kind='acceptable'` |
| `not_recommended` | Related terminology labeled "Not recommended" | `entity_related_terms.kind='not_recommended'` |
| `other` | Unlabeled related terminology, including recovered alternate terms | `entity_related_terms.kind='other'` |

A row with more than one origin combines distinct values alphabetically with
`; ` (for example `coding; contextual`). Filter with `LIKE` or split on `; `.

The frozen recovery manifest is `Library/recovered_terms.json` in the crawler
project. Full refresh re-applies it and preserves review status, qualifiers and
notes; mirroring includes both new tables. Initial recovery: 171 new term rows,
17 existing page-linked targets reused, 215 provenance links and 398 code matches
per database; 16 of 188 targets have no proposed match. Backups precede migration.


In [ ]:
# Comprehensive read-only output: retain uncoded terms and proposed alternatives.
from contextlib import closing
import bluebooks_local as local
with closing(local.connect_sqlite(local.ROOT / "bluebooks.db", readonly=True)) as recovery_db:
    print("Recovery tables:", {table: recovery_db.execute(
        f"SELECT COUNT(*) FROM {table}").fetchone()[0]
        for table in ("entity_term_provenance", "entity_term_code_matches")})
    recovery_rows = recovery_db.execute(
        "SELECT page_id,term,source_code,suggested_code,edition,official_term,"
        "match_type,suggested_qualifier,review_status "
        "FROM v_entity_term_code_matches WHERE source_kind='AI_recovered_prose' "
        "ORDER BY page_id,term,edition,candidate_order"
    ).fetchall()
    print("Recovered terms and all candidate matches:", recovery_rows[:30])
    page_code_terms = recovery_db.execute(
        "SELECT book_version,chapter,ch_order,page_title,page_url,page_sections,TermUS,"
                "TermNormUS,candidateTermUS,ICDO32,ICDO4,is_subtype AS deprecated_is_subtype,code_kind,source_kind,"
        "suggested_qualifier,match_type,candidate_topography_codes "
        "FROM v_page_code_terms ORDER BY book_id,ch_order,page_id,TermUS,code_edition,morphology_code LIMIT 30"
    ).fetchall()
    print("Distinct page code-term pairs:", page_code_terms)


## Step 0 — One-time setup

The crawler needs **Python 3.9+** and **Playwright** (the library that drives a
real browser). Step 6 uses **pandas** for quick tables, and the optional SQL
Server load uses **pyodbc**. Run the cell below once. It:

1. checks you're running a **local** kernel (warns if it looks remote),
2. installs `playwright`, `pandas`, and `pyodbc` if missing, and
3. downloads the **Chromium** browser Playwright uses.

The Chromium download is ~100–150 MB and only happens the first time.

> `pyodbc` covers the Python package. For Step 7 you still need a Microsoft
> SQL Server ODBC driver installed on Windows (typically Driver 17 or 18).

**Expected outcome:** kernel details print, required packages are present,
and Playwright Chromium finishes with `✅ Chromium ready.`

### Before you start (quick checklist)

- You are running this notebook on your local machine (not a remote kernel).
- You are comfortable keeping `auth_state.json` and `bluebooks.db` private.
- You expect a first-run Chromium download (~100–150 MB).
- SQL Server steps are optional; Steps 1–6 and Step 8 work with SQLite only.
- If you need SQL Server later, keep these references handy:
  - `SESSION_FIX_LOG_2026-07-21.md`
  - `SQLSERVER_CONNECTION_VARIANTS.md`

In [ ]:
# Cell 0.1

import sys, os, subprocess, platform, socket

print("Python:", sys.version.split()[0], "on", platform.system(), platform.release())

# --- gentle "are we local?" heuristic (informational only) ---
host = socket.gethostname().lower()
looks_remote = any(x in host for x in ("e2b", "sandbox", "runner", "colab", "container"))
if looks_remote or "COLAB_GPU" in os.environ:
    print("\n⚠️  This looks like a REMOTE/cloud kernel:", host)
    print("   The login + crawl steps need a LOCAL browser window and your BB session.")
    print("   Open this notebook in VS Code on your own computer before running Steps 3–5.")
else:
    print("Kernel host:", host, "(looks local — good)")

def _ensure(pkg, import_name=None):
    import importlib
    try:
        importlib.import_module(import_name or pkg)
        print(f"✅ {pkg} already installed")
    except ImportError:
        print(f"⏳ installing {pkg} …")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", pkg])
        print(f"✅ {pkg} installed")

_ensure("playwright")
_ensure("pandas")
_ensure("pyodbc")

def _run_streamed(cmd):
    """Run a command and stream its output into the notebook cell."""
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    try:
        for line in proc.stdout:
            print(line, end="", flush=True)
    finally:
        proc.wait()
    return proc.returncode

# Download the Chromium browser Playwright drives (idempotent — safe to re-run).
print("\n⏳ Ensuring Chromium is installed for Playwright …")
print("   (First run downloads ~150 MB — progress shown below, may take a minute)\n")
rc = _run_streamed([sys.executable, "-m", "playwright", "install", "chromium"])
if rc == 0:
    print("\n✅ Chromium ready.")
else:
    print(f"\n⚠️  playwright install exited with code {rc}.")
    print("   Run this once in a terminal:  python -m playwright install chromium")


Python: 3.14.6 on Windows 11
Kernel host: rlm-dell (looks local — good)
✅ playwright already installed
✅ pandas already installed
✅ pyodbc already installed

⏳ Ensuring Chromium is installed for Playwright …
   (First run downloads ~150 MB — progress shown below, may take a minute)


✅ Chromium ready.


### Where are the crawler scripts?

This notebook calls the crawler's Python scripts (`bluebooks_auth.py`,
`bluebooks_books.py`, `bluebooks_crawl.py`, `bluebooks_fts.py`, `search.py`).
By default it looks in the **same folder as this notebook**. If you keep the
scripts elsewhere, set `CRAWLER_DIR` below.

In [ ]:
# Cell 0.2

from pathlib import Path

# Folder containing the crawler .py scripts (default: this notebook's folder).
CRAWLER_DIR = Path(".").resolve()      # e.g. Path(r"C:\Users\you\bluebooks-crawler")

REQUIRED = ["bluebooks_auth.py", "bluebooks_books.py", "bluebooks_crawl.py",
            "bluebooks_fts.py", "search.py", "schema.sql"]
missing = [f for f in REQUIRED if not (CRAWLER_DIR / f).exists()]
if missing:
    print("❌ Missing crawler files in", CRAWLER_DIR)
    for m in missing:
        print("   -", m)
    print("\nSet CRAWLER_DIR to the folder that holds the crawler scripts, then re-run.")
else:
    print("✅ Found all crawler scripts in:", CRAWLER_DIR)

✅ Found all crawler scripts in: <BBCrawler Notebook>


### A small helper to run the crawler and show its output live

Every step below calls one crawler script. This `run_script()` helper runs it
and **streams the output into the notebook as it happens**, so you can watch a
long crawl progress. It returns the exit code (0 = success).

In [ ]:
# Cell 0.3

import os, subprocess, sys
from pathlib import Path

def run_script(script, *args, cwd=None):
    """Run a crawler script with live output. Returns the process exit code."""
    cwd = Path(cwd) if cwd else CRAWLER_DIR
    cmd = [sys.executable, str(cwd / script), *map(str, args)]
    print("▶", " ".join(cmd), "\n" + "-" * 70)
    env = os.environ.copy()
    # Force UTF-8 between parent/child so notebook output doesn't show mojibake.
    env.setdefault("PYTHONUTF8", "1")
    env.setdefault("PYTHONIOENCODING", "utf-8")
    proc = subprocess.Popen(cmd, cwd=str(cwd), stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, encoding="utf-8",
                            errors="replace", bufsize=1, env=env)
    try:
        for line in proc.stdout:
            print(line, end="")
    finally:
        proc.wait()
    print("-" * 70)
    print(("✅ done" if proc.returncode == 0 else f"⚠️ exited with code {proc.returncode}"),
          f"({script})")
    return proc.returncode

print("run_script() ready.")

run_script() ready.


## Step 1 — Configure the crawler (`config.json`)

The scripts read their settings from a `config.json` file. Edit the values
below and run the cell to write it. You rarely need to change more than
`db_path`.

| Setting | What it does |
|---|---|
| `base_url` | The Blue Books site root. Leave as-is. |
| `db_path` | Where the SQLite database is written (your corpus). |
| `auth_state_path` | Where your saved login session is stored (Step 3). |
| `headless` | `false` for login (a window must appear); the crawl itself can run headless. |
| `capture_images` | `true` stores image URLs; `false` = smaller/faster DB (links still kept). |
| `login_wait_seconds` | Fallback wait if you can't press Enter (non-interactive). |
| `nav_timeout_ms` | Per-navigation timeout. |
| `polite_delay_seconds` | Pause between page fetches (be a good citizen). |
| `keepalive_seconds` | Activity ping to fend off the ~15-min session timeout. |

**Expected outcome:** writes/updates `config.json` and echoes the settings.

In [ ]:
# Cell 1.1

import json
from pathlib import Path

config = {
    "base_url": "https://tumourclassification.iarc.who.int",
    "db_path": "bluebooks.db",
    "auth_state_path": "auth_state.json",
    "headless": True,             # Step 3 (auth) forces a visible window regardless
    "capture_images": True,       # set False for a smaller, faster DB
    "login_wait_seconds": 120,
    "nav_timeout_ms": 30000,
    "polite_delay_seconds": 1.0,
    "keepalive_seconds": 240,
}

CONFIG_PATH = CRAWLER_DIR / "config.json"
CONFIG_PATH.write_text(json.dumps(config, indent=2), encoding="utf-8")
print(f"✅ Wrote {CONFIG_PATH}")
print(json.dumps(config, indent=2))

✅ Wrote <BBCrawler Notebook>\config.json
{
  "base_url": "https://tumourclassification.iarc.who.int",
  "db_path": "bluebooks.db",
  "auth_state_path": "auth_state.json",
  "headless": true,
  "capture_images": true,
  "login_wait_seconds": 120,
  "nav_timeout_ms": 30000,
  "polite_delay_seconds": 1.0,
  "keepalive_seconds": 240
}


## Step 2 — Log in once and save your session

The crawler never sees your password. Instead you log in **once** in a real
browser window, and Playwright saves the resulting session (cookies) to
`auth_state.json`. The crawler then reuses that session headlessly.

**What happens when you run the next cell:**

1. A **Chromium window opens** on the BlueBooksOnline home page.
2. **Log in normally** (institution / username+password / SSO — whatever you use).
3. When you can see your subscribed content, **come back here** — the script is
   waiting. Because this runs inside the notebook, it can't read your keyboard,
   so it waits `login_wait_seconds` (default 120s) and then captures the session
   automatically. Make sure you're fully logged in before that timer ends.

> Sessions expire after ~15 minutes of inactivity. If a later step reports
> `auth_expired`, just run this cell again to refresh `auth_state.json`, then
> resume the crawl.

> If you'd rather press Enter yourself (more reliable), run this **in a terminal**
> instead: `python bluebooks_auth.py --config config.json`

In [ ]:
# Cell 2.1

# Do not run this cell because the code needs to capture the Enter key press in the CLI. 
# This Notebook page has no way to process the keypress.
# Run this in terminal, log into the website, come back to the terminal and press Enter.
# python.exe bluebooks_auth.py --config config.json 



# Opens a REAL browser window — must be run on your local machine.
# Give yourself time to finish logging in before login_wait_seconds elapses.
rc = run_script("bluebooks_auth.py", "--config", "config.json")
auth_file = CRAWLER_DIR / config["auth_state_path"]
if auth_file.exists():
    print(f"\n✅ Session saved to {auth_file} — you're ready to discover & crawl.")
else:
    print("\n⚠️ No auth_state file yet. Re-run this cell and finish logging in "
          "before the timer ends (or run bluebooks_auth.py in a terminal).")

▶ <PYTHON_EXECUTABLE> <BBCrawler Notebook>\bluebooks_auth.py --config config.json 
----------------------------------------------------------------------


## Step 3 — Discover the volumes your subscription exposes

`discover` opens the site (using your saved session) and harvests every book it
can see across all edition tabs (5th, 6th, Cytopathology / Reporting Systems),
saving them into the `books` table. Run it once; re-run only when new volumes
appear.

In [ ]:
# Cell 3.1

rc = run_script("bluebooks_books.py", "--config", "config.json", "discover")

▶ <PYTHON_EXECUTABLE> <BBCrawler Notebook>\bluebooks_books.py --config config.json discover 
----------------------------------------------------------------------
[discover] 5th: 0 candidate book(s) at https://tumourclassification.iarc.who.int/home
[discover] 5th: 14 visible card(s)
[discover] 6th: 0 candidate book(s) at https://tumourclassification.iarc.who.int/home
[discover] 6th: 2 visible card(s)
[discover] cytopathology: 0 candidate book(s) at https://tumourclassification.iarc.who.int/home
[discover] cytopathology: 4 visible card(s)
[discover] cytopathology: 0 candidate book(s) at https://tumourclassification.iarc.who.int/home
[discover] cytopathology: 4 visible card(s)
[discover] found and recorded 20 book(s).

  ID  Edition        Status  Pages  Pend  Err  Title
------------------------------------------------------------------------------------------
  31  5th            Print     156     9    0  Digestive System Tumours (5th ed.)
  32  5th            Print     105     0    0 

### List the inventory with crawl status

`list` prints every known book with its per-book status: **Pages** already
stored, **Pend**ing in the queue, and **Err**ors. Use this to decide what to
crawl next, and to watch progress between crawl runs.

Known IDs from earlier discovery (yours may differ):

| Edition | Book | ID |
|---|---|---|
| 5th | Digestive System | 31 |
| 5th | Central Nervous System | 45 |
| 5th | Genetic Tumour Syndromes (Beta) | 67 |
| 6th | Digestive System (Beta) | 72 |
| 6th | Breast (Beta) | 84 |
| Cytopathology | Lung Reporting System | 48 |

In [ ]:
# Cell 3.2

rc = run_script("bluebooks_books.py", "--config", "config.json", "list")

▶ <PYTHON_EXECUTABLE> <BBCrawler Notebook>\bluebooks_books.py --config config.json list 
----------------------------------------------------------------------

  ID  Edition        Status  Pages  Pend  Err  Title
------------------------------------------------------------------------------------------
  31  5th            Print     156     9    0  Digestive System Tumours (5th ed.)
  32  5th            Print     105     0    0  Breast Tumours (5th ed.)
  33  5th            Print     191     0    0  Soft Tissue and Bone Tumours (5th ed.)
  34  5th            Print     316     0    0  Female Genital Tumours (5th ed.)
  35  5th            Print     167     0    0  Thoracic Tumours (5th ed.)
  36  5th            Print     189     0    0  Urinary and Male Genital Tumours (5th ed.)
  44  5th            Print     347     0    0  Paediatric Tumours (5th ed.)
  45  5th            Print     138     0    0  Central Nervous System Tumours (5th ed.)
  52  5th            Print     271     0    0  

## Step 4 — Crawl the volume(s) you choose

Pick one or more book IDs in `BOOKS` below. The crawler harvests the table of
contents, queues every leaf page, then fetches and stores each one. It's
**resumable**: re-running picks up only `pending`/`error` pages, so it's cheap
to stop and continue.

**Options** (set the flags in the cell):

- `TOC_ONLY = True` — only harvest the TOC into the queue (no page fetching yet).
- `LIMIT = 5` — stop after N pages per book — perfect for a first **smoke test**.
- `NO_IMAGES = True` — skip image capture (smaller, faster DB; links still kept).
- `RESCRAPE = True` — purge this book's stored data and re-fetch it fresh
  (use when a Beta became Print, or an earlier crawl was incomplete).

**Start small.** Try one book with `LIMIT = 5` first to confirm everything works,
then re-run with `LIMIT = 0` to crawl it fully.

In [ ]:
# Cell 4.1

# ----------------------- choose what to crawl -----------------------
BOOKS     = [48,49,50,51]  # one or more book IDs, e.g. [31, 45, 72]
TOC_ONLY  = False       # True = only queue the TOC, don't fetch pages yet
LIMIT     = 0           # 0 = all pages; a small number = smoke test
NO_IMAGES = True        # True = skip images (smaller DB)
RESCRAPE  = False       # True = purge & re-fetch these books fresh
# --------------------------------------------------------------------

args = ["--config", "config.json", "--book", *map(str, BOOKS)]
if TOC_ONLY:  args.append("--toc-only")
if LIMIT:     args += ["--limit", str(LIMIT)]
if NO_IMAGES: args.append("--no-images")
if RESCRAPE:  args.append("--rescrape")

rc = run_script("bluebooks_crawl.py", *args)
if rc != 0:
    print("\nℹ️ If this said 'auth_expired', re-run Step 2 to refresh your session, "
          "then run this cell again — it resumes where it stopped.")

▶ <PYTHON_EXECUTABLE> <BBCrawler Notebook>\bluebooks_crawl.py --config config.json --book 64 --no-images 
----------------------------------------------------------------------
[plan] books=[64]  images=OFF  
[toc] book 64 — BlueBooksOnline: 287 leaf links, 287 queued.
  [1] done  editorial_board          https://tumourclassification.iarc.who.int/chaptercontent/64/392
  [2] done  how_to_cite              https://tumourclassification.iarc.who.int/chaptercontent/64/393
  [3] done  foreword                 https://tumourclassification.iarc.who.int/chaptercontent/64/394
  [4] done  icdo_coding              https://tumourclassification.iarc.who.int/chaptercontent/64/395
  [5] done  tnm_staging              https://tumourclassification.iarc.who.int/chaptercontent/64/420
  [6] done  other                    https://tumourclassification.iarc.who.int/chaptercontent/64/1
  [7] done  iccr_datasets            https://tumourclassification.iarc.who.int/chaptercontent/64/390
  [8] done  other        

**Full crawl after a good smoke test.** Once `LIMIT = 5` looks right, set
`LIMIT = 0` above and run the cell again to finish the book. Then re-run the
**Step 3 list** cell to watch `Pend` drop toward 0.

## Step 5 — Build the keyword search index (FTS5)

After crawling, build the full-text index so you can search the corpus. This
(re)builds two FTS5 tables from the stored sections: `sections_fts` (per-section)
and `pages_fts` (per-page). Safe to run repeatedly — run it again whenever you
crawl more pages.

In [ ]:
# Local-only enrichment: run after a crawl, or on the existing corpus.
from pathlib import Path
import bluebooks_local as local
DB = CRAWLER_DIR / config["db_path"]
print(local.backup_databases(DB, CRAWLER_DIR / "local_backups"))
local_result = local.refresh_local(DB)
print(local_result["counts"])
recovery = local_result["section_recovery"]
print({key: len(value) if isinstance(value, list) else value
       for key, value in recovery.items()})


## Step 6 — Check what you collected

A quick look inside the SQLite database: row counts per table, books with their
page counts, and a few sample pages with their clickable URLs. (This is
read-only.)

In [ ]:
# Cell 6.1

import sqlite3, pandas as pd
from pathlib import Path

DB = CRAWLER_DIR / config["db_path"]
print("Database:", DB, f"({DB.stat().st_size/1e6:.1f} MB)" if DB.exists() else "(not created yet)")
con = sqlite3.connect(DB)

def show(df, n=25):
    from IPython.display import display
    display(df.head(n))
    if len(df) > n: print(f"… {len(df)-n} more row(s).")

print("\nRow counts:")
import bluebooks_local as local
tables = local.TABLES
counts = []
for t in tables:
    try:
        (c,) = con.execute(f'SELECT COUNT(*) FROM "{t}"').fetchone()
    except Exception:
        c = "—"
    counts.append({"table": t, "rows": c})
show(pd.DataFrame(counts), n=50)

In [ ]:
# Cell 6.2

print("Books and how many pages are stored / pending / errored:")
show(pd.read_sql_query("""
    SELECT b.book_id, b.title, b.edition, b.status,
           SUM(p.crawl_status='done')    AS pages_done,
           SUM(p.crawl_status='pending') AS pending,
           SUM(p.crawl_status='error')   AS errors
    FROM books b LEFT JOIN pages p ON p.book_id = b.book_id
    GROUP BY b.book_id, b.title, b.edition, b.status
    ORDER BY b.book_id
""", con))

Books and how many pages are stored / pending / errored:


,book_id,title,edition,status,pages_done,pending,errors
0,31,Digestive System Tumours (5th ed.),5th,Print,156.0,0.0,0.0
1,32,Breast Tumours (5th ed.),5th,Print,105.0,0.0,0.0
2,33,Soft Tissue and Bone Tumours (5th ed.),5th,Print,191.0,0.0,0.0
3,34,Female Genital Tumours (5th ed.),5th,Print,316.0,0.0,0.0
4,35,Thoracic Tumours (5th ed.),5th,Print,167.0,0.0,0.0
5,36,Urinary and Male Genital Tumours (5th ed.),5th,Print,189.0,0.0,0.0
6,44,Paediatric Tumours (5th ed.),5th,Print,347.0,0.0,0.0
7,45,Central Nervous System Tumours (5th ed.),5th,Print,138.0,0.0,0.0
8,48,WHO Reporting System for Lung Cytopathology,cytopathology,Print,NaN,NaN,NaN
9,49,"WHO Reporting System for Lymph Node, Spleen, a...",cytopathology,Print,NaN,NaN,NaN


In [ ]:
# Cell 6.3

print("Sample crawled pages (with their Blue Books URLs):")
sample_pages = pd.read_sql_query("""
    SELECT p.page_id, b.edition, p.page_type, p.title, p.url
    FROM pages p JOIN books b ON b.book_id = p.book_id
    WHERE p.crawl_status = 'done'
    ORDER BY b.edition, p.page_type, p.title
    LIMIT 25
""", con)
with pd.option_context("display.max_colwidth", None):
    show(sample_pages)

Sample crawled pages (with their Blue Books URLs):


,page_id,edition,page_type,title,url
0,676,5th,chapter_intro,Introduction,https://tumourclassification.iarc.who.int/chaptercontent/67/1
1,4156,5th,chapter_intro,Introduction to B-cell lymphoproliferative disorders and neoplasms,https://tumourclassification.iarc.who.int/chaptercontent/63/287
2,2881,5th,chapter_intro,Introduction to CNS tumours,https://tumourclassification.iarc.who.int/chaptercontent/45/1
3,678,5th,chapter_intro,Introduction to Haematolymphoid Genetic Disorders,https://tumourclassification.iarc.who.int/chaptercontent/67/315
4,477,5th,chapter_intro,Introduction to eye and orbit tumours,https://tumourclassification.iarc.who.int/chaptercontent/65/97
5,4265,5th,chapter_intro,Introduction to primary cutaneous T-cell neoplasms,https://tumourclassification.iarc.who.int/chaptercontent/63/214
6,3291,5th,chapter_intro,Introduction to tumours of the breast,https://tumourclassification.iarc.who.int/chaptercontent/32/1
7,172,5th,chapter_intro,Introduction to tumours of the digestive system,https://tumourclassification.iarc.who.int/chaptercontent/31/1
8,2346,5th,chapter_intro,Introduction to urinary and male genital tumours,https://tumourclassification.iarc.who.int/chaptercontent/36/1
9,166,5th,editorial_board,WHO Classification of Tumours: Editorial Board,https://tumourclassification.iarc.who.int/chaptercontent/31/278


## Step 7 — (Optional) Copy the corpus into SQL Server

If you want the data in **SQL Server** (e.g. to run the cross-walk queries from
the *Explorer* notebook against a shared server), this step executes:

0. **rebuilds the SQLite FTS5 index** so the data you copy is fully indexed,
1. creates the schema from **`schema_sqlserver.sql`** (7a),
2. **deploys `dbo.Clean`** and verifies it (7a-clean),
3. **bulk-copies** every table from your finished SQLite DB into `dbo.*` (7b), and
4. **rebuilds the SQL Server FULLTEXT index** (7c).

### Preflight

- `pyodbc` installed in this Python environment.
- Microsoft **ODBC Driver 17/18 for SQL Server** installed.
- Target database already exists and you can write to it.
- SQL Server Full-Text Search feature is recommended for Step 7c `CONTAINS`.

### Execution order

Run these cells in order: **7·0 → 7a → 7a·clean → 7b → 7c**.

> This copies the **already-crawled** SQLite data — it does not re-crawl. Run
> Steps 4–5 first so the SQLite DB is complete.

**Expected outcome:** tables load into `dbo.*`, `dbo.Clean` verifies, and the
full-text catalog/index are rebuilt and populated.

### Rerun safety map

- Safe to rerun anytime: **Step 5, 7·0, 7a, 7a·clean, 7c, 8**.
- Usually rerun only when needed: **Step 2** (renew login), **Step 4** (crawl more content).
- After any large reload in 7b, rerun **7c** so SQL Server full-text search reflects new data.

### 7·0 — Rebuild the SQLite FTS5 index first

Before copying to SQL Server, make sure the **SQLite** search index reflects
everything you've crawled. This is the same rebuild as Step 5, repeated here so
the corpus you load is fully indexed (handy if you crawled more pages since you
last ran Step 5). Safe to run again — it rebuilds `sections_fts` / `pages_fts`
from scratch.

In [ ]:
# Cell 7.0.1

rc = run_script("bluebooks_fts.py", "--config", "config.json")
if rc == 0:
    print("✅ SQLite FTS5 index is current — ready to load SQL Server.")

▶ <PYTHON_EXECUTABLE> <BBCrawler Notebook>\bluebooks_fts.py --config config.json 
----------------------------------------------------------------------
[fts] indexed 25556 sections into sections_fts, 3195 pages into pages_fts.
----------------------------------------------------------------------
✅ done (bluebooks_fts.py)
✅ SQLite FTS5 index is current — ready to load SQL Server.


In [ ]:
# Cell 7.1

# ---------------------- SQL Server connection ----------------------
SQLSERVER = "(local)"                    # preferred server / instance (edit for your setup)
SQLSERVER_CANDIDATES = [
    SQLSERVER,
    "MACHINE_NAME",
    ".",
    "localhost",
]
DATABASE  = "BlueBooks"                  # target database (must already exist)
TRUSTED   = True                          # True = Windows login; False = SQL login
USERNAME  = ""                            # if TRUSTED = False
PASSWORD  = ""                            # leave blank to be prompted securely
DRIVER    = ""                            # blank = auto-pick 18 then 17 if installed
ENCRYPT   = True
# -------------------------------------------------------------------

def mssql_connect():
    import pyodbc, getpass

    def pick_driver():
        installed = [d.strip() for d in pyodbc.drivers()]
        if DRIVER:
            if DRIVER in installed:
                return DRIVER
            raise RuntimeError(f"Requested DRIVER not installed: {DRIVER}. Installed: {installed}")
        for d in ("ODBC Driver 18 for SQL Server", "ODBC Driver 17 for SQL Server"):
            if d in installed:
                return d
        raise RuntimeError("No SQL Server ODBC driver found. Install ODBC Driver 18 or 17.")

    def unique_servers(items):
        out = []
        seen = set()
        for s in items:
            s = str(s).strip()
            if s and s.lower() not in seen:
                seen.add(s.lower())
                out.append(s)
        return out

    pwd = PASSWORD
    if not TRUSTED and not pwd:
        pwd = getpass.getpass(f"Password for {USERNAME}@{SQLSERVER}: ")

    driver = pick_driver()
    attempts = []
    for server in unique_servers(SQLSERVER_CANDIDATES):
        parts = [f"DRIVER={{{driver}}}", f"SERVER={server}", f"DATABASE={DATABASE}"]
        parts.append("Trusted_Connection=yes" if TRUSTED else f"UID={USERNAME};PWD={pwd}")
        parts.append("Encrypt=yes" if ENCRYPT else "Encrypt=no")
        parts.append("TrustServerCertificate=yes")
        parts.append("Connection Timeout=5")
        try:
            cn = pyodbc.connect(";".join(parts), autocommit=False)
            globals()["SQLSERVER"] = server
            print(f"✅ Connected to SQL Server using: {server} (driver: {driver})")
            return cn
        except Exception as e:
            attempts.append(f"{server}: {str(e).splitlines()[0][:140]}")

    raise RuntimeError(
        "Could not connect to any configured SQL Server target. Tried:\n- "
        + "\n- ".join(attempts)
        + "\nEdit SQLSERVER or SQLSERVER_CANDIDATES in this cell, then re-run it."
    )

print("Configured SQL Server target:", SQLSERVER, "/", DATABASE)
print("Run the next cell to create the schema, then the one after to copy data.")

Configured SQL Server target: (local) / BlueBooks
Run the next cell to create the schema, then the one after to copy data.


### 7a — Create the SQL Server schema

Runs `schema_sqlserver.sql` (splitting on `GO` batches). Existing tables are
left alone (`IF OBJECT_ID(...) IS NULL`), so it's safe to re-run.

**Expected outcome:** `✅ Applied X/Y schema batches` (with possible informative
"already exists" skips on reruns).

If this fails: first re-run the SQL Server connection cell above, then re-run 7a.

In [ ]:
# Bootstrap the complete data schema, not activity metadata.
import bluebooks_local as local
with local.connect_sqlite(CRAWLER_DIR / config["db_path"]) as src, local.connect_server(SQLSERVER, DATABASE) as dst:
    local.migrate_sqlite(src)
    local.ensure_server_schema(src, dst)
print("Complete local data schema ready.")


### 7a·clean — Deploy `dbo.Clean` and verify it

The PathCHART cross-walk (in the *Explorer* notebook) and any term matching on
the server rely on the **`dbo.Clean`** normalization function — the SQL Server
twin of the Python `clean()` used everywhere else. This cell deploys the
**authoritative** function body from **`00_dbo_Clean.sql`** (the corrected
2026-07-09 version, mirrored 1:1 by `python/clean_term.py`) using
`CREATE OR ALTER`, so it's safe to run repeatedly.

In [ ]:
# Cell 7a.2

import re

def split_tsql_batches(sql_text: str):
    # Remove /* ... */ blocks first so 'GO' inside comments does not split batches.
    no_block_comments = re.sub(r"/\*.*?\*/", "", sql_text, flags=re.S)
    return [b.strip() for b in re.split(r"(?im)^\s*GO\s*$", no_block_comments) if b.strip()]

clean_path = CRAWLER_DIR / "00_dbo_Clean.sql"
if not clean_path.exists():
    print("⚠️ 00_dbo_Clean.sql not found next to the notebook.")
    print("   Copy it from the term-normalizer bundle (sqlserver/00_dbo_Clean.sql) and re-run.")
else:
    body = clean_path.read_text(encoding="utf-8")
    batches = split_tsql_batches(body)
    cn = mssql_connect(); cur = cn.cursor()
    ok = 0
    for b in batches:
        try:
            cur.execute(b); ok += 1
        except Exception as e:
            print("⚠️ batch failed:", str(e).splitlines()[0][:160])
    cn.commit(); cn.close()
    print(f"✅ Deployed dbo.Clean ({ok}/{len(batches)} batch(es) executed).")

✅ Connected to SQL Server using: (local) (driver: ODBC Driver 18 for SQL Server)
✅ Deployed dbo.Clean (1/1 batch(es) executed).


**Verify `dbo.Clean` exists and behaves.** This confirms the function is
present on the server and runs a few normalization smoke tests (US spelling,
`NOS`/`type` stripping, `/`→space, Roman-numeral handling). Compare the output
to what the Python `clean()` produces — they must match.

In [ ]:
# Cell 7a.3

cn = mssql_connect(); cur = cn.cursor()

# 1) Does the scalar function exist?
cur.execute("SELECT OBJECT_ID('dbo.Clean', 'FN')")
if cur.fetchone()[0] is None:
    print("❌ dbo.Clean is NOT present on the server. Run the deploy cell above first.")
else:
    print("✅ dbo.Clean exists on", f"{SQLSERVER}/{DATABASE}.")
    # 2) Smoke tests: (input, expected)
    tests = [
        ("Oesophageal adenocarcinoma, NOS", "Esophageal adenocarcinoma"),
        ("Follicular lymphoma, duodenal-type", "Follicular lymphoma duodenal"),
        ("AIN III",                            "AIN 3"),
        ("Adenocarcinoma/NOS",                 "Adenocarcinoma"),
    ]
    print(f"\n{'input':45} {'dbo.Clean output':35} {'expected':35} ok?")
    print("-" * 125)
    all_ok = True
    for src, exp in tests:
        cur.execute("SELECT dbo.Clean(?)", src)
        got = cur.fetchone()[0]
        ok = (got == exp)
        all_ok &= ok
        print(f"{src:45} {str(got):35} {exp:35} {'✓' if ok else '✗'}")
    print("-" * 125)
    print("✅ All smoke tests passed." if all_ok else
          "⚠️ Some outputs differ — confirm this is the corrected dbo.Clean "
          "(re-deploy from 00_dbo_Clean.sql).")
cn.close()

✅ Connected to SQL Server using: (local) (driver: ODBC Driver 18 for SQL Server)
✅ dbo.Clean exists on (local)/BlueBooks.

input                                         dbo.Clean output                    expected                            ok?
-----------------------------------------------------------------------------------------------------------------------------
Oesophageal adenocarcinoma, NOS               Esophageal adenocarcinoma           Esophageal adenocarcinoma           ✓
Follicular lymphoma, duodenal-type            Follicular lymphoma duodenal        Follicular lymphoma duodenal        ✓
AIN III                                       AIN 3                               AIN 3                               ✓
Adenocarcinoma/NOS                            Adenocarcinoma                      Adenocarcinoma                      ✓
-----------------------------------------------------------------------------------------------------------------------------
✅ All smoke tests passe

### 7b — Copy the data (SQLite ➜ SQL Server)

Streams each table from SQLite into `dbo.<table>` using batched `INSERT`s
(`fast_executemany` for speed). Set `TRUNCATE_FIRST = True` to clear the target
tables before copying (a clean reload). Column names are matched by name, so
extra/missing columns are handled gracefully.

> After copying, rebuild the SQL Server FULLTEXT index if needed (see
> `schema_sqlserver.sql`), and deploy `dbo.Clean` from the *term-normalizer*
> bundle if you'll run the PathCHART cross-walk here.

In [ ]:
# Mirror approved data only; agent activity and crawl bookkeeping are untouched.
import bluebooks_local as local
DB = CRAWLER_DIR / config["db_path"]
print(local.backup_databases(DB, CRAWLER_DIR / "local_backups", SQLSERVER, DATABASE))
parity = local.mirror(DB, SQLSERVER, DATABASE, include_book70=False)
print({table: result["match"] for table, result in parity.items()})


### 7c — Rebuild the SQL Server FULLTEXT index

SQL Server's full-text index does **not** repopulate itself when you bulk-load
rows the way FTS5 is rebuilt in Step 5. After a reload you must (re)create and
**fully populate** the index over `dbo.sections(heading, body_text)` so
`CONTAINS(...)` / the `dbo.v_search` view return the new content.

The cell below does this robustly:

1. checks the **Full-Text Search** feature is installed,
2. creates the `BlueBooksFT` catalog if missing,
3. looks up the **real primary-key index name** on `dbo.sections` (SQL Server
   auto-names it `PK__sections__…`, so the literal `PK__sections` placeholder in
   the schema won't match) — this is the required unique key index,
4. **drops any existing** full-text index and recreates it against that key, and
5. forces a **full population** and reports catalog status.

> Requires the SQL Server **Full-Text Search** component. If it's not installed,
> the cell says so and everything else (exact/`LIKE` queries, the cross-walk,
> the Explorer notebook's FTS on SQLite) still works — only server-side
> `CONTAINS` needs it.

**Expected outcome:** index recreation message, population status, and a
`CONTAINS` smoke-test result (`0 row(s)` is valid if the term is absent).

If this fails: run 7a first, confirm Full-Text Search is installed, then rerun 7c.

In [ ]:
# Verify section/title/whole-page FULLTEXT; fail explicitly on timeout.
import bluebooks_local as local
with local.connect_server(SQLSERVER, DATABASE) as cn:
    print(local.server_search(cn))
    print(local.search(cn, "carcinoma", backend="sqlserver", limit=3))


### 7c.2 — Optional: Add SQL Server FULLTEXT on page titles

Use this only if you want direct `CONTAINS` queries on `dbo.pages.title`.

This cell:

1. checks Full-Text Search is installed,
2. ensures the `BlueBooksFT` catalog exists,
3. finds the real primary-key index on `dbo.pages`,
4. drops/recreates a `dbo.pages(title)` full-text index, and
5. runs a quick `CONTAINS` smoke test.

Safe to rerun. It only affects the full-text index on `dbo.pages`.

In [ ]:
# Verify section/title/whole-page FULLTEXT; fail explicitly on timeout.
import bluebooks_local as local
with local.connect_server(SQLSERVER, DATABASE) as cn:
    print(local.server_search(cn))
    print(local.search(cn, "carcinoma", backend="sqlserver", limit=3))


## Step 8 — Quick search sanity check

Confirm the index works by running the crawler's own search CLI. Change the
query to anything in your corpus.

**Expected outcome:** printed hits and `✅ done (search.py)`.

Notes:
- `0 hit(s)` can still be healthy if the query term is not present.
- Output is UTF-8 normalized and snippet text is cleaned to remove common site-chrome noise.

In [8]:
# Cell 8.1

QUERY = "neuroendocrine"

# Self-heal for partial notebook reruns where the helper cell was skipped.
if "run_script" not in globals():
    import os, subprocess, sys
    from pathlib import Path

    def run_script(script, *args, cwd=None):
        cwd = Path(cwd) if cwd else CRAWLER_DIR
        cmd = [sys.executable, str(cwd / script), *map(str, args)]
        print("▶", " ".join(cmd), "\n" + "-" * 70)
        env = os.environ.copy()
        env.setdefault("PYTHONUTF8", "1")
        env.setdefault("PYTHONIOENCODING", "utf-8")
        proc = subprocess.Popen(
            cmd, cwd=str(cwd), stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, encoding="utf-8", errors="replace", bufsize=1, env=env
        )
        try:
            for line in proc.stdout:
                print(line, end="")
        finally:
            proc.wait()
        print("-" * 70)
        print(("✅ done" if proc.returncode == 0 else f"⚠️ exited with code {proc.returncode}"),
              f"({script})")
        return proc.returncode

rc = run_script("search.py", "--config", "config.json", QUERY, "--limit", "10")

▶ <PYTHON_EXECUTABLE> <BBCrawler Notebook>\search.py --config config.json neuroendocrine --limit 10 
----------------------------------------------------------------------

■ Mixed neuroendocrine-non-neuroendocrine neoplasms
  Endocrine and Neuroendocrine Tumours (5th ed.) (5th) · other
  https://tumourclassification.iarc.who.int/chaptercontent/53/124
  Mixed neuroendocrine-non-neuroendocrine neoplasms: …

■ Pancreatic mixed neuroendocrine–non-neuroendocrine neoplasms
  Digestive System Tumours (6th ed.) (6th) · other
  https://tumourclassification.iarc.who.int/chaptercontent/72/238
  Pancreatic mixed neuroendocrine–non-neuroendocrine neoplasms: …

■ Neuroendocrine tumour
  Breast Tumours (6th ed.) (6th) · other
  https://tumourclassification.iarc.who.int/chaptercontent/84/98
  Neuroendocrine tumour: …

■ Neuroendocrine carcinoma
  Breast Tumours (6th ed.) (6th) · other
  https://tumourclassification.iarc.who.int/chaptercontent/84/99
  Neuroendocrine carcinoma: …

■ Neuroendocrine tumo

### What next?

Choose your next path:

- Need more corpus data? Set `BOOKS` in **Step 4**, run it, then re-run **Step 5**.
- Need SQL Server analytics/search? Follow **7·0 → 7a → 7a·clean → 7b → 7c**.
- Need query exploration/cross-walks? Open **`BlueBooks_Explorer.ipynb`** and point it at `bluebooks.db` (or SQL Server).
- Need troubleshooting context from this session?
  - `SESSION_FIX_LOG_2026-07-21.md`
  - `SQLSERVER_CONNECTION_VARIANTS.md`

Always keep `auth_state.json` and `bluebooks.db` **private** (session + licensed content).